# Parameter Sensitivity Analysis (Continuous & Rule-Based)

Single notebook for both **continuous** and **rule-based** features. Set `feature_type` in
`feature_research/config.py` (or override below); the same workflow runs:

- **Continuous:** Sweeps feature params + `n_bins`, fits `ContinuousBinningModel`, computes metric on selected returns.
- **Rule-based:** Sweeps feature params only; feature is already a binary signal — no model fitting.

Both paths run neighbor smoothing → floor-based stable region selection → recommendations and plots.

**Workflow:**
1. Set `feature_type` in `feature_research/config.py` (CONTINUOUS or RULE_BASED); override inline if needed.
2. Run all cells (Kernel → Restart & Run All).
3. Interact with the Plotly figures in Section 5.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import replace

from feature_research.config import FeatureType
from feature_research.in_sample.config import load_config
from feature_research.in_sample.metric_helpers import compute_param_sensitivity_metric
from feature_research.in_sample.data_loader import (
    expand_bias_specs,
    load_features_for_combo,
    populate_cache_if_needed,
    param_combo_label,
)
from feature_selection.base_models.continuous_binning import ContinuousBinningModel
from eda.parameter_analysis import generate_parameter_sensitivity_report
from utils.core.enums import Ticker, TimeFrame

print("Imports OK")


Imports OK


## 1. Configuration

Edit `config.py` for persistent changes, or use the override cell below for one-off runs.
`dataclasses.replace()` is used because `ResearchConfig` is a frozen dataclass.

In [2]:
# --- Load defaults from config.py ---
config = load_config()

# ============================================================
# OVERRIDE CELL — edit here to customise without touching config.py
# ============================================================

# config = replace(config, tickers=[Ticker.ES])
# config = replace(config, start=datetime(2000, 1, 1), end=datetime(2015, 12, 31))
# Switch to rule-based without editing feature_research/config.py:
# config = replace(config, feature_type=FeatureType.RULE_BASED)
# config = replace(config,
#     bias_spec={
#         "module_name": "rsi",
#         "timeframes": [TimeFrame.D],
#         "params": {"lookback": [5, 10, 14, 20, 30]},
#     }
# )

# --- Param sensitivity: from config (edit feature_research/config.py to tune) ---
ps = config.param_sensitivity
USE_FLOOR_BASED_SELECTION = ps.use_floor_based_selection
STABILITY_THRESHOLD = ps.stability_threshold
# top_k is global (GlobalResearchDefaults); walkforward is built from it
TOP_K = config.walkforward.top_k
BINNING_SELECTION_METRIC = config.binning_params.selection_metric
METRIC_COL = BINNING_SELECTION_METRIC
PLOT_3D_MODE = ps.plot_3d_mode
# To change top_k, edit GlobalResearchDefaults in feature_research/config.py load_config().

# --- Dispatch on feature_type (from feature_research/config.py) ---
is_continuous = config.feature_type == FeatureType.CONTINUOUS
if is_continuous:
    N_BINS_GRID = list(config.binning_params.bin_counts)
    if not N_BINS_GRID:
        raise ValueError(
            "binning_params.bin_counts is empty. Provide at least one n_bins value "
            "for parameter sensitivity."
        )
else:
    N_BINS_GRID = None  # rule-based: fixed 3 levels, no n_bins sweep

# Derive varying param names (list-valued params only)
varying_params = [
    k for k, v in config.bias_spec["params"].items()
    if isinstance(v, list) and len(v) > 1
]
if not varying_params:
    print(
        "WARNING: No list-valued params found in bias_spec. "
        "Parameter sensitivity requires at least one param with multiple values. "
        "Results will be a single-point grid for feature params; only n_bins will vary."
    )
    varying_params = list(config.bias_spec["params"].keys())

report_param_names = [*varying_params, "n_bins", "selected_bin"] if is_continuous else varying_params

src = ps.stable_region_config
print(f"Feature type : {config.feature_type.value}")
print(f"Module      : {config.bias_spec['module_name']}")
print(f"Tickers     : {[t.name for t in config.tickers]}")
print(f"Period      : {config.start.date()} -> {config.end.date()}")
print(f"Varying     : {varying_params}")
if is_continuous:
    print(f"n_bins grid : {N_BINS_GRID}")
print(f"METRIC      : {METRIC_COL}  |  THRESHOLD: {STABILITY_THRESHOLD}")
print(f"BIN METRIC  : {BINNING_SELECTION_METRIC}")
print(f"3D MODE     : {PLOT_3D_MODE}")
print(f"Selection   : floor_based={USE_FLOOR_BASED_SELECTION}  top_k={TOP_K}  floor={src.floor_method}  k_per_region={src.k_per_region}  k_max={src.k_max}  min_region_size={src.min_region_size}")



Feature type : rule_based
Module      : rsi_signal
Tickers     : ['ES', 'NQ', 'RTY']
Period      : 2000-01-01 -> 2023-12-30
Varying     : ['rsi_period', 'oversold', 'overbought']
METRIC      : t_stat  |  THRESHOLD: 0.7
BIN METRIC  : t_stat
3D MODE     : surface_slices
Selection   : floor_based=True  top_k=1  floor=adaptive  k_per_region=3  k_max=3  min_region_size=3


## 2. Feature Extraction

Loads feature + target data for every param combo. Uses the cache if `config.use_cache=True`.
Set `config.populate_cache=True` to auto-populate missing cache entries.

In [3]:
populate_cache_if_needed(config)

expanded_specs = expand_bias_specs(config.bias_spec)
print(f"Parameter grid: {len(expanded_specs)} combinations\n")

combo_data = {}
for spec in expanded_specs:
    result = load_features_for_combo(spec, config)
    if result is not None:
        feature, target, feature_col = result
        combo_data[param_combo_label(spec["params"])] = {
            "feature": feature,
            "target": target,
            "params": spec["params"],
        }
    else:
        print(f"  SKIP: {spec['params']}")

print(f"\nLoaded {len(combo_data)}/{len(expanded_specs)} combos successfully")

[data_loader] Populating cache with full OHLC range: 1997-09-09 -> 2025-09-19
  Adding required auxiliary: atr (for log_return_atr/log_return_ewsd)
  Adding required auxiliary: ewsd (for log_return_atr/log_return_ewsd)
Populating cache for 384 combinations...
  Bias node specs: 128 (including required atr/ewsd if needed)
  Tickers: 3
  Date range: 1997-09-09 00:00:00 to 2025-09-19 00:00:00
  Max workers: 4
  [1/384] rsi_signal / ES / D
  [2/384] rsi_signal / ES / D
  [3/384] rsi_signal / RTY / D
  [4/384] rsi_signal / NQ / D
  [5/384] rsi_signal / NQ / D
  [6/384] rsi_signal / ES / D
  [7/384] rsi_signal / RTY / D
  [8/384] rsi_signal / RTY / D
  [9/384] rsi_signal / NQ / D
  [10/384] rsi_signal / RTY / D
  [11/384] rsi_signal / ES / D
  [12/384] rsi_signal / NQ / D
  [13/384] rsi_signal / NQ / D
  [14/384] rsi_signal / ES / D
  [15/384] rsi_signal / ES / D
  [16/384] rsi_signal / RTY / D
  [17/384] rsi_signal / NQ / D
  [18/384] rsi_signal / RTY / D
  [19/384] rsi_signal / NQ / D
  [2

## 3. Metric Grid

- **Continuous:** Fits `ContinuousBinningModel` for each `(feature params, n_bins)` combo; then for each bin index in `0 .. n_bins-1` computes the config-defined metric on returns in that bin. Emits one row per `(feature params, n_bins, selected_bin)` so the grid is 3D — enabling stable-region selection to favor regions where bin 0 (tail/mean reversion) is strong.
- **Rule-based:** Feature is already a binary signal (1 = trade); computes the config-defined binning metric directly on returns where signal fires — no model fitting.

Builds `results_df` with `param{K}_value` columns (`param1`=varying params, `param2`=n_bins, `param3`=selected_bin for continuous) for `generate_parameter_sensitivity_report()`.


In [4]:
rows = []

for spec in expanded_specs:
    label = param_combo_label(spec["params"])
    if label not in combo_data:
        continue

    data = combo_data[label]
    feature = data["feature"]
    target = data["target"]

    if is_continuous:
        for n_bins in N_BINS_GRID:
            try:
                model = ContinuousBinningModel(n_bins=n_bins)
                model.fit(feature, target)
                bin_assignments = model._assign_bins(feature)
                for bin_index in range(n_bins):
                    mask = (bin_assignments == bin_index)
                    selected_returns = target.loc[mask].values
                    if len(selected_returns) < 5:
                        continue
                    metric_value = compute_param_sensitivity_metric(
                        selected_returns, BINNING_SELECTION_METRIC
                    )
                    row = {}
                    for k, name in enumerate(report_param_names):
                        if name == "n_bins":
                            row[f"param{k+1}_value"] = n_bins
                        elif name == "selected_bin":
                            row[f"param{k+1}_value"] = bin_index
                        else:
                            row[f"param{k+1}_value"] = spec["params"][name]
                    row[METRIC_COL] = metric_value
                    rows.append(row)
            except Exception as e:
                print(f"  ERROR [{label}, n_bins={n_bins}]: {e}")
    else:
        try:
            signals = np.asarray(feature.values == 1).flatten()
            selected_returns = target.values[signals]

            if len(selected_returns) < 5:
                print(f"  SKIP [{label}]: too few signals ({len(selected_returns)}) to compute reliable metric")
                continue

            metric_value = compute_param_sensitivity_metric(selected_returns, BINNING_SELECTION_METRIC)
            row = {f"param{k+1}_value": spec["params"][key] for k, key in enumerate(varying_params)}
            row[METRIC_COL] = metric_value
            rows.append(row)

        except Exception as e:
            print(f"  ERROR [{label}]: {e}")

results_df = pd.DataFrame(rows)
print(f"Grid results: {len(results_df)} rows")
results_df


Grid results: 126 rows


,param1_value,param2_value,param3_value,t_stat
0,2,5,95,0.585522
1,2,5,90,0.460215
2,2,5,85,0.439860
3,2,5,80,0.515742
4,2,5,75,0.664520
...,...,...,...,...
121,4,30,85,1.654955
122,4,30,80,1.625159
123,4,30,75,1.622904
124,4,30,70,1.748361


## 4. Parameter Sensitivity Report

Runs: neighbor smoothing → stable region detection → recommendations.

In [5]:
if results_df.empty:
    raise RuntimeError(
        f"No results collected. Check that combos loaded: "
        f"{len(combo_data)}/{len(expanded_specs)} loaded. "
        f"Ensure cache is available or set config.populate_cache=True."
    )

report = generate_parameter_sensitivity_report(
    results_df=results_df,
    param_names=report_param_names,
    metric_col=METRIC_COL,
    stability_threshold=STABILITY_THRESHOLD,
    top_k=TOP_K,
    plot_3d_mode=PLOT_3D_MODE,
    use_floor_based_selection=USE_FLOOR_BASED_SELECTION,
    stable_region_config=config.param_sensitivity.stable_region_config,
)

print(f"Module              : {config.bias_spec['module_name']}")
print(f"Metric              : {report.metric_name}")
print(f"Grid size           : {report.n_parameter_combinations}")
print(f"Stable regions      : {report.n_stable_regions}")
print(f"Mean stability ratio: {report.mean_stability_ratio:.3f}")
print(f"% stable combos     : {report.pct_stable_combinations:.1%}")
print(f"\nTop-{TOP_K} recommendations:")
for i, combo in enumerate(report.top_k_combinations, 1):
    params_str = ", ".join(
        f"{name}={val}" for name, val in zip(report_param_names, combo)
    )
    print(f"  {i}. {params_str}")




Module              : rsi_signal
Metric              : t_stat
Grid size           : 126
Stable regions      : 1
Mean stability ratio: 2.319
% stable combos     : 5.6%

Top-1 recommendations:
  1. rsi_period=2.0, oversold=30.0, overbought=95.0


### Stable regions and summary statistics

Summary stats from the selection algorithm and a table of all identified stable regions (contiguous grid regions above the stability threshold).

In [6]:
# --- Summary statistics ---
grid = report.grid_results
ratios = grid["stability_ratio"].dropna()
raw_metric = report.metric_name
smoothed_metric = f"smoothed_{raw_metric}"
obj_vals = grid[raw_metric].dropna() if raw_metric in grid.columns else pd.Series(dtype=float)
smoothed_vals = grid[smoothed_metric].dropna() if smoothed_metric in grid.columns else pd.Series(dtype=float)

summary = pd.DataFrame([
    {"Metric": "Objective metric", "Value": report.metric_name},
    {"Metric": "Stability threshold", "Value": report.stability_threshold},
    {"Metric": "Grid size (param combos)", "Value": report.n_parameter_combinations},
    {"Metric": "Stable regions", "Value": report.n_stable_regions},
    {"Metric": "Total combos in stable regions", "Value": sum(r.n_combinations for r in report.stable_regions)},
    {"Metric": "Mean stability ratio", "Value": f"{report.mean_stability_ratio:.3f}"},
    {"Metric": "Median stability ratio", "Value": f"{report.median_stability_ratio:.3f}"},
    {"Metric": "Std stability ratio", "Value": f"{ratios.std():.3f}" if len(ratios) > 1 else "—"},
    {"Metric": "Min / Max stability ratio", "Value": f"{ratios.min():.3f} / {ratios.max():.3f}" if len(ratios) > 0 else "—"},
    {"Metric": "% combos in stable regions", "Value": f"{report.pct_stable_combinations:.1%}"},
    {"Metric": f"Best raw {raw_metric}", "Value": f"{obj_vals.max():.4f}" if len(obj_vals) > 0 else "—"},
    {"Metric": f"Best smoothed {raw_metric}", "Value": f"{smoothed_vals.max():.4f}" if len(smoothed_vals) > 0 else "—"},
    {"Metric": f"25th / 75th pct raw {raw_metric}", "Value": f"{obj_vals.quantile(0.25):.4f} / {obj_vals.quantile(0.75):.4f}" if len(obj_vals) > 0 else "—"},
    {"Metric": f"25th / 75th pct smoothed {raw_metric}", "Value": f"{smoothed_vals.quantile(0.25):.4f} / {smoothed_vals.quantile(0.75):.4f}" if len(smoothed_vals) > 0 else "—"},
]).set_index("Metric")
display(summary)

# --- All stable regions (param ranges, size, objective stats) ---
if not report.stable_regions:
    print("No stable regions with ≥2 combinations identified.")
else:
    region_rows = [
        {
            "region": i + 1,
            "param_ranges": ", ".join(
                f"{report_param_names[k]}={r.param_ranges[f'param{k+1}'][0]}..{r.param_ranges[f'param{k+1}'][1]}"
                for k in range(len(report_param_names))
            ),
            "n_combinations": r.n_combinations,
            "mean_objective": round(r.mean_objective, 4),
            "min_objective": round(r.min_objective, 4),
            "max_objective": round(r.max_objective, 4),
            "mean_stability_ratio": round(r.mean_stability_ratio, 3),
            "is_boundary": r.is_boundary_region,
        }
        for i, r in enumerate(report.stable_regions)
    ]
    regions_df = pd.DataFrame(region_rows)
    display(regions_df)

    # --- Per-region param combos with per-combo stats (raw, smoothed, stability_ratio, n_neighbors) ---
    param_cols = [f"param{k+1}_value" for k in range(len(report_param_names))]
    smoothed_col = f"smoothed_{report.metric_name}"
    print("\nParam combos per region (with stats from grid):")
    for i, r in enumerate(report.stable_regions):
        combos_df = pd.DataFrame(
            [list(t) for t in r.param_combinations],
            columns=report_param_names,
        )
        merged = combos_df.merge(
            report.grid_results,
            left_on=report_param_names,
            right_on=param_cols,
            how="left",
        )
        stats_cols = [report.metric_name, smoothed_col, "stability_ratio", "n_neighbors"]
        keep = report_param_names + [c for c in stats_cols if c in merged.columns]
        out = merged[keep].copy()
        round_map = {c: 4 for c in [report.metric_name, smoothed_col] if c in out.columns}
        if "stability_ratio" in out.columns:
            round_map["stability_ratio"] = 3
        if round_map:
            out = out.round(round_map)
        renames = {}
        if report.metric_name in out.columns:
            renames[report.metric_name] = f"{report.metric_name}_raw"
        if smoothed_col in out.columns:
            renames[smoothed_col] = f"{report.metric_name}_smoothed"
        if renames:
            out = out.rename(columns=renames)
        print(f"\nRegion {i + 1} ({r.n_combinations} combos):")
        display(out)

,Value
Metric,
Objective metric,t_stat
Stability threshold,0.7
Grid size (param combos),126
Stable regions,1
Total combos in stable regions,7
Mean stability ratio,2.319
Median stability ratio,1.006
Std stability ratio,22.176
Min / Max stability ratio,-146.962 / 190.653


,region,param_ranges,n_combinations,mean_objective,min_objective,max_objective,mean_stability_ratio,is_boundary
0,1,"rsi_period=2.0..3.0, oversold=25.0..30.0, over...",7,2.4902,2.348,2.6409,0.967,True



Param combos per region (with stats from grid):

Region 1 (7 combos):


,rsi_period,oversold,overbought,t_stat_raw,t_stat_smoothed,stability_ratio,n_neighbors
0,2.0,25.0,85.0,2.5159,2.3925,0.951,5
1,2.0,25.0,90.0,2.7002,2.5292,0.937,5
2,2.0,25.0,95.0,2.7764,2.5815,0.930,4
3,2.0,30.0,85.0,2.3544,2.3899,1.015,4
4,2.0,30.0,90.0,2.6225,2.5498,0.972,4
5,2.0,30.0,95.0,2.7802,2.6409,0.950,3
6,3.0,25.0,95.0,2.3147,2.3480,1.014,5


## 5. Interactive Visualizations

The appropriate plot is selected automatically based on grid dimensionality:
- 1D grid → raw + smoothed line with shaded stable regions
- 2D grid → heatmap with stability contour overlay
- 3D+ grid → heatmap slices with dropdown + slider controls

In [7]:
if report.plot_1d is not None:
    report.plot_1d.show()

if report.plot_2d is not None:
    report.plot_2d.show()

if report.plot_3d is not None:
    report.plot_3d.show()

## 6. Optional Save

Uncomment cells below to save plots as interactive HTML or grid results as parquet.

In [8]:
# --- OPTIONAL: Save outputs (uncomment to enable) ---

# from pathlib import Path
# save_dir = config.reports_dir / "param_sensitivity"
# save_dir.mkdir(parents=True, exist_ok=True)

# if report.plot_1d is not None:
#     report.plot_1d.write_html(save_dir / f"param_sensitivity_1d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 1D plot -> {save_dir}")

# if report.plot_2d is not None:
#     report.plot_2d.write_html(save_dir / f"param_sensitivity_2d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 2D plot -> {save_dir}")

# if report.plot_3d is not None:
#     report.plot_3d.write_html(save_dir / f"param_sensitivity_3d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 3D plot -> {save_dir}")

# report.grid_results.to_parquet(save_dir / "param_sensitivity_grid.parquet")
# print(f"Saved grid results -> {save_dir / 'param_sensitivity_grid.parquet'}")